In [1]:
import pandas as pd
import numpy as np

import re    

from nltk.tokenize import word_tokenize           
from nltk.corpus import stopwords                 
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory 
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\zulfi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
df_thr = pd.read_excel('../datasets/01_eda.xlsx')
df_thr.head()

,platform,text
0,tiktok,asa te boga walikota..hanya KDM yg ku punya
1,tiktok,sedih gara gara bapak ini
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k..."
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...
4,tiktok,nyesel milih dia


# **2. PREPROCESSING**

## **2.1 Cleaing Data**

In [3]:
# Handling missing value
print(f'Sebelum handling missing value : \n{df_thr.isna().sum()}\n')

df_thr.dropna(inplace=True)
print(f"Setelah handling missing value : \n{df_thr.isna().sum()}")

Sebelum handling missing value : 
platform     0
text        43
dtype: int64

Setelah handling missing value : 
platform    0
text        0
dtype: int64


## **2.2 Cleaning Text**

In [4]:
# Fungsi hapus URL
def remove_URL(tweet):
    if tweet is not None and isinstance(tweet, str):
        url = re.compile(r'https?://\S+|www\.\S+')
        return url.sub(r'', tweet)
    else:
        return tweet

# Fungsi hapus HTML
def remove_html(tweet):
    if tweet is not None and isinstance(tweet, str):
        html = re.compile(r'<.*?>')
        return html.sub(r'', tweet)
    else:
        return tweet

# Fungsi hapus emoji
def remove_emoji(tweet):
    if tweet is not None and isinstance(tweet, str):
        emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"  # emoticons
            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # transport & map symbols
            u"\U0001F700-\U0001F77F"  # alchemical symbols
            u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
            u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
            u"\U0001FA00-\U0001FA6F"  # Chess Symbols
            u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
            u"\U0001F004-\U0001F0CF"  # Additional emoticons
            u"\U0001F1E0-\U0001F1FF"  # flags
                               "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', tweet)
    else:
        return tweet

# Fungsi hampus simbol
def remove_symbols(tweet):
    if tweet is not None and isinstance(tweet, str):
        tweet = re.sub(r'[^a-zA-Z0-9\s]', '', tweet)
    return tweet

# Fungsi hapus username
def remove_usernames(text):
    if text is not None and isinstance(text, str):
        return re.sub(r'@\w+', '', text)
    else:
        return text

df_thr['cleaning'] = df_thr['text'].apply(lambda x: remove_URL(x))
df_thr['cleaning'] = df_thr['cleaning'].apply(lambda x: remove_html(x))
df_thr['cleaning'] = df_thr['cleaning'].apply(lambda x: remove_usernames(x))
df_thr['cleaning'] = df_thr['cleaning'].apply(lambda x: remove_emoji(x))
df_thr['cleaning'] = df_thr['cleaning'].apply(lambda x: remove_symbols(x))

df_thr.head()

,platform,text,cleaning
0,tiktok,asa te boga walikota..hanya KDM yg ku punya,asa te boga walikotahanya KDM yg ku punya
1,tiktok,sedih gara gara bapak ini,sedih gara gara bapak ini
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k...",hade goreng ku basa ieu mah Pareum we eweuh kl...
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...,Ciamis ts cair Minggu kmari Tasik mah kmha cer...
4,tiktok,nyesel milih dia,nyesel milih dia


## **2.3 Case Folding**

In [5]:
def case_folding(text):
    if isinstance(text, str):
        lowercase_text = text.lower()
        return lowercase_text
    else:
        return text

df_thr['case_folding'] = df_thr['cleaning'].apply(case_folding)
df_thr.head()

,platform,text,cleaning,case_folding
0,tiktok,asa te boga walikota..hanya KDM yg ku punya,asa te boga walikotahanya KDM yg ku punya,asa te boga walikotahanya kdm yg ku punya
1,tiktok,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k...",hade goreng ku basa ieu mah Pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...,Ciamis ts cair Minggu kmari Tasik mah kmha cer...,ciamis ts cair minggu kmari tasik mah kmha cer...
4,tiktok,nyesel milih dia,nyesel milih dia,nyesel milih dia


## **2.4 Normalization**

In [6]:
# URL RAW GitHub
url = "https://raw.githubusercontent.com/analisis25/data_kamus_tb/main/kamus_kata_tidak_baku.csv"

# Baca CSV langsung
kamus_data = pd.read_csv(url, sep=';')
print("Kolom:", kamus_data.columns.tolist())

# Hapus "ku" dari kamus normalisasi Indonesia
kamus_data = kamus_data[
    kamus_data['Kata Tidak Baku'].str.lower() != 'ku'
]

# Membuat dictionary
kamus_tidak_baku_dict = dict(
    zip(kamus_data['Kata Tidak Baku'], kamus_data['Kata Baku'])
)

print(f"Jumlah kata dalam kamus: {len(kamus_tidak_baku_dict)}")

Kolom: ['Kata Tidak Baku', 'Kata Baku']
Jumlah kata dalam kamus: 8867


In [7]:
# Normalisasi kata

def normalisasi_kata(text):
    if pd.isna(text):
        return text

    words = text.split()

    normalized_words = [
        kamus_tidak_baku_dict.get(word, word)
        for word in words
    ]

    return ' '.join(normalized_words)

df_thr['normalisasi_kata'] = df_thr['case_folding'].apply(normalisasi_kata)
df_thr.head(20)

,platform,text,cleaning,case_folding,normalisasi_kata
0,tiktok,asa te boga walikota..hanya KDM yg ku punya,asa te boga walikotahanya KDM yg ku punya,asa te boga walikotahanya kdm yg ku punya,asa te boga walikotahanya kdm yang ku punya
1,tiktok,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k...",hade goreng ku basa ieu mah Pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...,Ciamis ts cair Minggu kmari Tasik mah kmha cer...,ciamis ts cair minggu kmari tasik mah kmha cer...,ciamis ts cair minggu kmari tasik mah kmha cer...
4,tiktok,nyesel milih dia,nyesel milih dia,nyesel milih dia,menyesal memilih dia
5,tiktok,terlalu maksakeun jadi walikota,terlalu maksakeun jadi walikota,terlalu maksakeun jadi walikota,terlalu maksakeun jadi walikota
6,tiktok,ke1,ke1,ke1,ke1
7,tiktok,asa kkara tahun ieu walikota ku Viman jadi kieu,asa kkara tahun ieu walikota ku Viman jadi kieu,asa kkara tahun ieu walikota ku viman jadi kieu,asa kkara tahun ieu walikota ku viman jadi kieu
8,tiktok,2,2,2,2
9,tiktok,Parahhh,Parahhh,parahhh,parahhh


In [8]:
tambahan_kamus = {
    'belumpadahal': 'belum padahal', 'ts': 'atos', 'kmari': 'kamari',
    'walikota': 'wali kota', 'te': 'henteu', 't': 'henteu', 'hnt': 'henteu',
    'hnte': 'henteu', 'hnteu': 'henteu', 'hent': 'henteu', 'tt': 'henteu',
    'kkara': 'kakara', 'walikotahanya': 'wali kota, hanya', 'tsm': 'tasikmalaya',
    'taskot': 'kota tasikmalaya', 'tasik': 'tasikmalaya', 'thrdi': 'thr di',
    'heulangke': 'heula engke', 'ahhcan': 'ahh acan', 'pguh': 'puguh',
    'daekn': 'daekeun', 'yata': 'nyata', 'heulakakara': 'heula kakara',
    'thrulah': 'thr ulah', 'mtak': 'matak', 'ulh': 'ulah', 'vimaan': 'viman',
    'mni': 'meni', 'trsb': 'tersebut', 'ieung': 'ieu', 'sejaw': 'sejawa',
    'kekanpung2': 'ke kampung kampung', 'cikurubukancurr': 'cikurubuk ancur',
    'propinsikota': 'provinsi kota', 'kabupatensami': 'kabupaten sami', 
    'hnjakal': 'hanjakal', 'walikotateh': 'wali kota teh', 'kmha': 'kumaha',
    'knh' : 'keneh', 'jalaan': 'jalan', 'dkt': 'deukeut',
    'logaktara': 'logak tara', 'snes': 'sanes', 'manehnamah': 'maneh na mah',
    'walikotana': 'wali kota na', 'pnten': 'punten', 'sshacuma': 'sasaha cuma',
    'thunn': 'tahun', 'kuacis': 'ku acis', 'ce pe': 'cepe'
}
kamus_tidak_baku_dict.update(tambahan_kamus)

df_thr['normalisasi_kata'] = df_thr['case_folding'].apply(normalisasi_kata)
df_thr.head(20)

,platform,text,cleaning,case_folding,normalisasi_kata
0,tiktok,asa te boga walikota..hanya KDM yg ku punya,asa te boga walikotahanya KDM yg ku punya,asa te boga walikotahanya kdm yg ku punya,"asa henteu boga wali kota, hanya kdm yang ku p..."
1,tiktok,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k...",hade goreng ku basa ieu mah Pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...,Ciamis ts cair Minggu kmari Tasik mah kmha cer...,ciamis ts cair minggu kmari tasik mah kmha cer...,ciamis atos cair minggu kamari tasikmalaya mah...
4,tiktok,nyesel milih dia,nyesel milih dia,nyesel milih dia,menyesal memilih dia
5,tiktok,terlalu maksakeun jadi walikota,terlalu maksakeun jadi walikota,terlalu maksakeun jadi walikota,terlalu maksakeun jadi wali kota
6,tiktok,ke1,ke1,ke1,ke1
7,tiktok,asa kkara tahun ieu walikota ku Viman jadi kieu,asa kkara tahun ieu walikota ku Viman jadi kieu,asa kkara tahun ieu walikota ku viman jadi kieu,asa kakara tahun ieu wali kota ku viman jadi kieu
8,tiktok,2,2,2,2
9,tiktok,Parahhh,Parahhh,parahhh,parahhh


## **2.5 Tokenizing**

In [9]:
def tokenize(text):
    tokens = text.split()
    return tokens

df_thr['tokenize'] = df_thr['normalisasi_kata'].apply(tokenize)
df_thr.head()

,platform,text,cleaning,case_folding,normalisasi_kata,tokenize
0,tiktok,asa te boga walikota..hanya KDM yg ku punya,asa te boga walikotahanya KDM yg ku punya,asa te boga walikotahanya kdm yg ku punya,"asa henteu boga wali kota, hanya kdm yang ku p...","[asa, henteu, boga, wali, kota,, hanya, kdm, y..."
1,tiktok,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,"[sedih, gara, gara, bapak, ini]"
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k...",hade goreng ku basa ieu mah Pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,"[hade, goreng, ku, basa, ieu, mah, pareum, we,..."
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...,Ciamis ts cair Minggu kmari Tasik mah kmha cer...,ciamis ts cair minggu kmari tasik mah kmha cer...,ciamis atos cair minggu kamari tasikmalaya mah...,"[ciamis, atos, cair, minggu, kamari, tasikmala..."
4,tiktok,nyesel milih dia,nyesel milih dia,nyesel milih dia,menyesal memilih dia,"[menyesal, memilih, dia]"


## **2.6 Stopword**

In [10]:
nltk.download('stopwords')
stop_words = stopwords.words('indonesian')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zulfi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
def remove_stopwords(text):
    return [word for word in text if word not in stop_words]

# Ubah hasil list jadi string
df_thr['stopword removal'] = df_thr['tokenize'].apply(lambda x: " ".join(remove_stopwords(x)))

df_thr.head()

,platform,text,cleaning,case_folding,normalisasi_kata,tokenize,stopword removal
0,tiktok,asa te boga walikota..hanya KDM yg ku punya,asa te boga walikotahanya KDM yg ku punya,asa te boga walikotahanya kdm yg ku punya,"asa henteu boga wali kota, hanya kdm yang ku p...","[asa, henteu, boga, wali, kota,, hanya, kdm, y...","asa henteu boga wali kota, kdm ku"
1,tiktok,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,"[sedih, gara, gara, bapak, ini]",sedih gara gara
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k...",hade goreng ku basa ieu mah Pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,"[hade, goreng, ku, basa, ieu, mah, pareum, we,...",hade goreng ku basa ieu mah pareum we eweuh kl...
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...,Ciamis ts cair Minggu kmari Tasik mah kmha cer...,ciamis ts cair minggu kmari tasik mah kmha cer...,ciamis atos cair minggu kamari tasikmalaya mah...,"[ciamis, atos, cair, minggu, kamari, tasikmala...",ciamis atos cair minggu kamari tasikmalaya mah...
4,tiktok,nyesel milih dia,nyesel milih dia,nyesel milih dia,menyesal memilih dia,"[menyesal, memilih, dia]",menyesal memilih


## **2.7 Stemming**

In [12]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming_text(text):
    if isinstance(text, str):
        return stemmer.stem(text)
    else:
        return text

In [13]:
df_thr['stemming'] = df_thr['stopword removal'].apply(stemming_text)
df_thr[['stopword removal', 'stemming']].head(10)

,stopword removal,stemming
0,"asa henteu boga wali kota, kdm ku",asa henteu boga wali kota kdm ku
1,sedih gara gara,sedih gara gara
2,hade goreng ku basa ieu mah pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...
3,ciamis atos cair minggu kamari tasikmalaya mah...,ciamis atos cair minggu kamar tasikmalaya mah ...
4,menyesal memilih,sesal pilih
5,maksakeun wali kota,maksakeun wali kota
6,ke1,ke1
7,asa kakara ieu wali kota ku viman kieu,asa kakara ieu wali kota ku viman kieu
8,2,2
9,parahhh,parahhh


In [14]:
df_thr.head()

,platform,text,cleaning,case_folding,normalisasi_kata,tokenize,stopword removal,stemming
0,tiktok,asa te boga walikota..hanya KDM yg ku punya,asa te boga walikotahanya KDM yg ku punya,asa te boga walikotahanya kdm yg ku punya,"asa henteu boga wali kota, hanya kdm yang ku p...","[asa, henteu, boga, wali, kota,, hanya, kdm, y...","asa henteu boga wali kota, kdm ku",asa henteu boga wali kota kdm ku
1,tiktok,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,sedih gara gara bapak ini,"[sedih, gara, gara, bapak, ini]",sedih gara gara,sedih gara gara
2,tiktok,"hade goreng ku basa, ieu mah Pareum we eweuh k...",hade goreng ku basa ieu mah Pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...,"[hade, goreng, ku, basa, ieu, mah, pareum, we,...",hade goreng ku basa ieu mah pareum we eweuh kl...,hade goreng ku basa ieu mah pareum we eweuh kl...
3,tiktok,Ciamis ts cair Minggu kmari .Tasik mah kmha ce...,Ciamis ts cair Minggu kmari Tasik mah kmha cer...,ciamis ts cair minggu kmari tasik mah kmha cer...,ciamis atos cair minggu kamari tasikmalaya mah...,"[ciamis, atos, cair, minggu, kamari, tasikmala...",ciamis atos cair minggu kamari tasikmalaya mah...,ciamis atos cair minggu kamar tasikmalaya mah ...
4,tiktok,nyesel milih dia,nyesel milih dia,nyesel milih dia,menyesal memilih dia,"[menyesal, memilih, dia]",menyesal memilih,sesal pilih


In [15]:
df_thr.to_excel('../datasets/02_preprocessing.xlsx', index=False)